In [1]:
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option("display.max_colwidth", None)

In [ ]:
# import ijson
# import pandas as pd

# file_path = "/home/user/Downloads/bayut_properties_rent_commercial_2026_08_19.json"

# with open(file_path, "rb") as f:
#     data = list(ijson.items(f, "item"))

# df = pd.DataFrame(data)

# print(df.shape)


(28634, 111)


In [23]:
import ijson
import pandas as pd
import gc

file_path = "/home/user/Downloads/bayut_properties_buy_residential_2026_08_19.json"

CHUNK_SIZE = 5000

with open(file_path, "rb") as f:
    items = ijson.items(f, "item")

    chunk = []

    for item in items:
        chunk.append(item)

        if len(chunk) == CHUNK_SIZE:

            df = pd.DataFrame(chunk)

            print("Chunk shape:", df.shape)

            # =========================
            # YOUR QA CHECKS HERE
            # =========================

            # Example
            print(df.columns.tolist())

            # Check every column
            for column in df.columns:
                print("\nColumn:", column)
                print(df[column].head())

            # =========================

            chunk.clear()
            del df
            gc.collect()

    # Process remaining records
    if chunk:
        df = pd.DataFrame(chunk)

        print("Final chunk shape:", df.shape)

        # QA checks here

        del df
        gc.collect()

Chunk shape: (5000, 107)
['listing_url', 'id', 'objectID', 'ownerID', 'userExternalID', 'sourceID', 'state', 'geography', 'purpose', 'price', 'product', 'productLabel', 'productVariant', 'rentFrequency', 'referenceNumber', 'projectNumber', 'title', 'title_l1', 'title_l2', 'title_l3', 'title_l4', 'title_l5', 'title_l6', 'title_l7', 'title_l8', 'title_l9', 'title_l10', 'title_l11', 'title_l12', 'title_l13', 'title_l14', 'description', 'description_l1', 'description_l2', 'description_l3', 'description_l4', 'description_l5', 'description_l6', 'description_l7', 'description_l8', 'description_l9', 'description_l10', 'description_l11', 'description_l12', 'description_l13', 'description_l14', 'descriptionTranslated', 'descriptionTranslated_l1', 'descriptionTranslated_l2', 'descriptionTranslated_l3', 'externalID', 'slug', 'location', 'category', 'createdAt', 'approvedAt', 'updatedAt', 'touchedAt', 'reactivatedAt', 'rooms', 'baths', 'area', 'score', 'score_l1', 'score_l2', 'score_l3', 'coverPhot

In [24]:

import json
import ijson
import pandas as pd
import requests
from urllib.parse import urlparse
from datetime import datetime
import random
 
FILE_PATH = "/home/user/Downloads/bayut_properties_buy_residential_2026_08_19.json"
EXPECTED_COUNT = 138781
RANDOM_CHECK_COUNT = 10
 
print("✓ Setup complete")
print(f"File: {FILE_PATH}")
print(f"Expected count: {EXPECTED_COUNT:,}")

✓ Setup complete
File: /home/user/Downloads/bayut_properties_buy_residential_2026_08_19.json
Expected count: 138,781


In [25]:
print("\n" + "="*80)
print("CHECK 1: JSON VALIDITY")
print("="*80)
 
try:
    with open(FILE_PATH, 'rb') as f:
        first_item = ijson.items(f, "item")
        test_item = next(first_item)
    
    print("✅ JSON is valid")
    print(f"✅ First item parsed successfully")
    print(f"✅ Item type: {type(test_item).__name__}")
    print(f"✅ Item has {len(test_item)} fields")
    
except json.JSONDecodeError as e:
    print(f"❌ JSON is INVALID: {str(e)}")
except Exception as e:
    print(f"❌ Error reading JSON: {str(e)}")


CHECK 1: JSON VALIDITY
✅ JSON is valid
✅ First item parsed successfully
✅ Item type: dict
✅ Item has 98 fields


In [26]:
print("\n" + "="*80)
print("CHECK 2: RECORD COUNT VERIFICATION")
print("="*80)
 
count = 0
with open(FILE_PATH, 'rb') as f:
    for _ in ijson.items(f, "item"):
        count += 1
 
print(f"Actual count: {count:,}")
print(f"Expected count: {EXPECTED_COUNT:,}")
 
if count == EXPECTED_COUNT:
    print(f"✅ COUNT MATCHES! ({count:,})")
else:
    diff = count - EXPECTED_COUNT
    print(f"❌ COUNT MISMATCH!")
    print(f"   Difference: {diff:+,} ({(diff/EXPECTED_COUNT*100):+.2f}%)")



CHECK 2: RECORD COUNT VERIFICATION
Actual count: 118,611
Expected count: 138,781
❌ COUNT MISMATCH!
   Difference: -20,170 (-14.53%)


In [27]:
print("\n" + "="*80)
print("LOADING SMALL SAMPLE (first 100 items only)")
print("="*80)
 
sample_items = []
with open(FILE_PATH, 'rb') as f:
    for idx, item in enumerate(ijson.items(f, "item")):
        sample_items.append(item)
        if idx >= 99:  # Just first 100
            break
 
print(f"✓ Loaded {len(sample_items)} sample items")
 
# Create small DataFrame just for this sample
df_sample = pd.DataFrame(sample_items)
print(f"✓ Created sample DataFrame: {df_sample.shape}")
 
print(f"\nSample DataFrame info:")
print(f"  Rows: {df_sample.shape[0]}")
print(f"  Columns: {df_sample.shape[1]}")
print(f"  Fields: {', '.join(df_sample.columns.tolist()[:8])}")


LOADING SMALL SAMPLE (first 100 items only)
✓ Loaded 100 sample items
✓ Created sample DataFrame: (100, 104)

Sample DataFrame info:
  Rows: 100
  Columns: 104
  Fields: listing_url, id, objectID, ownerID, userExternalID, sourceID, state, geography


In [28]:
print("\n" + "="*80)
print("CHECK 3: MANDATORY FIELDS (Stream validation)")
print("="*80)
 
# Track missing values without loading all data
field_stats = {}
 
with open(FILE_PATH, 'rb') as f:
    for idx, item in enumerate(ijson.items(f, "item")):
        # First item: initialize field_stats
        if idx == 0:
            for field in item.keys():
                field_stats[field] = {'present': 0, 'missing': 0}
        
        # Count presence for each field
        for field in field_stats.keys():
            if field in item and item[field] is not None:
                field_stats[field]['present'] += 1
            else:
                field_stats[field]['missing'] += 1
        
        # Progress
        if (idx + 1) % 50000 == 0:
            print(f"  Checked {idx+1:,} items...")
 
print(f"\nField Completeness (based on {idx+1:,} items):")
print("─" * 80)
 
all_complete = True
for field in sorted(field_stats.keys()):
    stats = field_stats[field]
    total = stats['present'] + stats['missing']
    missing_pct = (stats['missing'] / total * 100) if total > 0 else 0
    
    if stats['missing'] == 0:
        status = "✅"
    else:
        status = "⚠️"
        all_complete = False
    
    print(f"{status} {field:<30} Present: {stats['present']:>7,} | Missing: {stats['missing']:>7,} ({missing_pct:>6.2f}%)")
 
print("\n" + "─" * 80)
if all_complete:
    print("✅ ALL FIELDS ARE COMPLETE (No missing values)")
else:
    print("⚠️  Some fields have missing values")


CHECK 3: MANDATORY FIELDS (Stream validation)
  Checked 50,000 items...
  Checked 100,000 items...

Field Completeness (based on 118,611 items):
────────────────────────────────────────────────────────────────────────────────
✅ active                         Present: 118,611 | Missing:       0 (  0.00%)
✅ agency                         Present: 118,611 | Missing:       0 (  0.00%)
✅ agentAdStoriesCount            Present: 118,611 | Missing:       0 (  0.00%)
✅ amenities                      Present: 118,611 | Missing:       0 (  0.00%)
✅ approvedAt                     Present: 118,611 | Missing:       0 (  0.00%)
✅ area                           Present: 118,611 | Missing:       0 (  0.00%)
✅ baths                          Present: 118,611 | Missing:       0 (  0.00%)
✅ category                       Present: 118,611 | Missing:       0 (  0.00%)
✅ cityLevelScore                 Present: 118,611 | Missing:       0 (  0.00%)
✅ completionStatus               Present: 118,611 | Missing:  

In [29]:
print("\n" + "="*80)
print("CHECK 7: LOCATION VERIFICATION (Dubai only)")
print("="*80)
 
location_stats = {
    'dubai': 0,
    'non_dubai': 0,
    'missing': 0,
    'samples': []
}
 
with open(FILE_PATH, 'rb') as f:
    for idx, item in enumerate(ijson.items(f, "item")):
        # Find location field
        location = None
        for loc_field in ['location', 'area', 'emirate']:
            if loc_field in item:
                location = item[loc_field]
                break
        
        if location is None:
            location_stats['missing'] += 1
        elif 'dubai' in str(location).lower():
            location_stats['dubai'] += 1
            if len(location_stats['samples']) < 5:
                location_stats['samples'].append(location)
        else:
            location_stats['non_dubai'] += 1
        
        # Progress
        if (idx + 1) % 50000 == 0:
            print(f"  Checked {idx+1:,} items...")
 
print(f"\nLocation Analysis (based on {idx+1:,} items):")
print("─" * 80)
print(f"✅ Dubai items: {location_stats['dubai']:,}")
print(f"❌ Non-Dubai items: {location_stats['non_dubai']:,}")
print(f"⚠️  Missing location: {location_stats['missing']:,}")
 
if location_stats['non_dubai'] == 0:
    print("\n✅ ALL ITEMS ARE FROM DUBAI ONLY!")
else:
    print(f"\n❌ CRITICAL: Found {location_stats['non_dubai']} non-Dubai items")
 
print(f"\nSample Dubai locations:")
for loc in location_stats['samples']:
    print(f"  • {loc}")


CHECK 7: LOCATION VERIFICATION (Dubai only)
  Checked 50,000 items...
  Checked 100,000 items...

Location Analysis (based on 118,611 items):
────────────────────────────────────────────────────────────────────────────────
✅ Dubai items: 118,611
❌ Non-Dubai items: 0
⚠️  Missing location: 0

✅ ALL ITEMS ARE FROM DUBAI ONLY!

Sample Dubai locations:
  • [{'id': 1, 'level': 0, 'externalID': '5001', 'name': 'UAE', 'name_l1': 'الإمارات', 'name_l2': '阿联酋', 'name_l3': 'ОАЭ', 'slug': '/uae'}, {'id': 2, 'level': 1, 'externalID': '5002', 'name': 'Dubai', 'name_l1': 'دبي', 'name_l2': '迪拜', 'name_l3': 'Дубай', 'slug': '/dubai'}, {'id': 59, 'level': 2, 'externalID': '5416', 'name': 'Jumeirah Village Circle (JVC)', 'name_l1': 'قرية جميرا الدائرية', 'name_l2': '朱美拉环形村(JVC)', 'name_l3': 'Джумейра Вилладж Серкл (ДЖВС)', 'slug': '/dubai/jumeirah-village-circle-jvc', 'type': 'neighbourhood'}, {'id': 1935, 'level': 3, 'externalID': '11385', 'name': 'JVC District 13', 'name_l1': 'الضاحية 13', 'name_l2': '

In [30]:
import ijson
import pandas as pd
import csv
 
FILE_PATH = "/home/user/Downloads/bayut_properties_buy_residential_2026_08_19.json"
OUTPUT_CSV = "missing_coverphoto_records.csv"
 
print("\n" + "="*80)
print("FINDING MISSING COVERPHOTO RECORDS")
print("="*80)
 
missing_records = []
 
with open(FILE_PATH, 'rb') as f:
    for record_num, item in enumerate(ijson.items(f, "item"), 1):
        # Check if coverPhoto is missing or null
        if 'coverPhoto' not in item or item['coverPhoto'] is None or item['coverPhoto'] == '':
            
            # Get listing_url
            listing_url = item.get('listing_url', 'N/A')
            product_id = item.get('id', 'N/A')
            title = item.get('title', 'N/A')
            
            missing_records.append({
                'record_number': record_num,
                'product_id': product_id,
                'title': title,
                'listing_url': listing_url
            })
            
            print(f"Record {record_num}: Missing coverPhoto")
            print(f"  ID: {product_id}")
            print(f"  URL: {listing_url}")
            print()
 
print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"Total missing coverPhoto records: {len(missing_records)}")
 
# Export to CSV
if missing_records:
    df = pd.DataFrame(missing_records)
    df.to_csv(OUTPUT_CSV, index=False)
    print(f"✅ Exported to: {OUTPUT_CSV}")
    
    print("\nRecords found:")
    print(df.to_string(index=False))
else:
    print("No missing records found")


FINDING MISSING COVERPHOTO RECORDS
Record 42967: Missing coverPhoto
  ID: 12279660
  URL: https://www.bayut.com/property/details-15756475.html

Record 65538: Missing coverPhoto
  ID: 12199874
  URL: https://www.bayut.com/property/details-15674716.html

Record 74176: Missing coverPhoto
  ID: 12614888
  URL: https://www.bayut.com/property/details-16107981.html

Record 87000: Missing coverPhoto
  ID: 7533746
  URL: https://www.bayut.com/property/details-10663495.html

Record 91126: Missing coverPhoto
  ID: 12396226
  URL: https://www.bayut.com/property/details-15879454.html

Record 103669: Missing coverPhoto
  ID: 12632502
  URL: https://www.bayut.com/property/details-16126071.html

Record 112675: Missing coverPhoto
  ID: 11950280
  URL: https://www.bayut.com/property/details-15419439.html


SUMMARY
Total missing coverPhoto records: 7
✅ Exported to: missing_coverphoto_records.csv

Records found:
 record_number  product_id                                                                   

In [31]:
print("\n" + "="*80)
print("CHECK 5 & 6: LISTING_URL VALIDATION")
print("="*80)
 
url_stats = {
    'total': 0,
    'present': 0,
    'missing': 0,
    'valid': 0,
    'invalid': 0,
    'bayut': 0,
    'non_bayut': 0,
    'invalid_examples': []
}
 
with open(FILE_PATH, 'rb') as f:
    for idx, item in enumerate(ijson.items(f, "item")):
        url_stats['total'] += 1
        
        if 'listing_url' not in item or item['listing_url'] is None:
            url_stats['missing'] += 1
        else:
            url_stats['present'] += 1
            url = str(item['listing_url']).strip()
            
            # Check URL format
            try:
                result = urlparse(url)
                if all([result.scheme in ['http', 'https'], result.netloc]):
                    url_stats['valid'] += 1
                else:
                    url_stats['invalid'] += 1
                    if len(url_stats['invalid_examples']) < 3:
                        url_stats['invalid_examples'].append(url[:100])
            except:
                url_stats['invalid'] += 1
            
            # Check Bayut domain
            if 'bayut' in url.lower():
                url_stats['bayut'] += 1
            else:
                url_stats['non_bayut'] += 1
        
        # Progress
        if (idx + 1) % 50000 == 0:
            print(f"  Checked {idx+1:,} items...")
 
print(f"\nListing URL Analysis (based on {url_stats['total']:,} items):")
print("─" * 80)
print(f"Present: {url_stats['present']:,}")
print(f"Missing: {url_stats['missing']:,}")
print(f"\nURL Format:")
print(f"  ✅ Valid: {url_stats['valid']:,}")
print(f"  ❌ Invalid: {url_stats['invalid']:,}")
if url_stats['invalid_examples']:
    print(f"  Examples of invalid URLs:")
    for url in url_stats['invalid_examples']:
        print(f"    {url}")
 
print(f"\nBayut Domain:")
print(f"  ✅ Bayut URLs: {url_stats['bayut']:,}")
print(f"  ❌ Non-Bayut URLs: {url_stats['non_bayut']:,}")
 
if url_stats['missing'] == 0 and url_stats['invalid'] == 0 and url_stats['non_bayut'] == 0:
    print("\n✅ ALL LISTING URLS ARE VALID BAYUT URLS!")
else:
    print("\n⚠️  Some URL issues found")


CHECK 5 & 6: LISTING_URL VALIDATION
  Checked 50,000 items...
  Checked 100,000 items...

Listing URL Analysis (based on 118,611 items):
────────────────────────────────────────────────────────────────────────────────
Present: 118,611
Missing: 0

URL Format:
  ✅ Valid: 118,611
  ❌ Invalid: 0

Bayut Domain:
  ✅ Bayut URLs: 118,611
  ❌ Non-Bayut URLs: 0

✅ ALL LISTING URLS ARE VALID BAYUT URLS!


In [32]:

print("\n" + "="*80)
print("CHECK 8: DATA TYPE VERIFICATION (Rent vs Sale)")
print("="*80)
 
type_stats = {
    'rent': 0,
    'sale': 0,
    'other': 0,
    'missing': 0
}
 
with open(FILE_PATH, 'rb') as f:
    for idx, item in enumerate(ijson.items(f, "item")):
        # Find transaction type field
        trans_type = None
        for type_field in ['type', 'property_type', 'category', 'transaction_type', 'purpose']:
            if type_field in item:
                trans_type = item[type_field]
                break
        
        if trans_type is None:
            type_stats['missing'] += 1
        else:
            type_str = str(trans_type).lower()
            if 'rent' in type_str or 'lease' in type_str:
                type_stats['rent'] += 1
            elif 'sale' in type_str or 'buy' in type_str or 'purchase' in type_str:
                type_stats['sale'] += 1
            else:
                type_stats['other'] += 1
        
        # Progress
        if (idx + 1) % 50000 == 0:
            print(f"  Checked {idx+1:,} items...")
 
print(f"\nData Type Analysis (based on {idx+1:,} items):")
print("─" * 80)
print(f"✅ Rent items: {type_stats['rent']:,}")
print(f"❌ Sale items: {type_stats['sale']:,}")
print(f"⚠️  Other items: {type_stats['other']:,}")
print(f"Missing type: {type_stats['missing']:,}")
 
if type_stats['sale'] == 0 and type_stats['other'] == 0:
    print("\n✅ ALL ITEMS ARE RENT ONLY! (No sales)")
elif type_stats['sale'] > 0:
    print(f"\n❌ CRITICAL: Found {type_stats['sale']} sale items (should be 0)")


CHECK 8: DATA TYPE VERIFICATION (Rent vs Sale)
  Checked 50,000 items...
  Checked 100,000 items...

Data Type Analysis (based on 118,611 items):
────────────────────────────────────────────────────────────────────────────────
✅ Rent items: 0
❌ Sale items: 0
⚠️  Other items: 118,611
Missing type: 0


In [34]:

import ijson
from collections import Counter
 
FILE_PATH = "/home/user/Downloads/bayut_properties_buy_residential_2026_08_19.json"
 
print("\n" + "="*80)
print("CHECKING 'PURPOSE' FIELD VALUES")
print("="*80)
 
purpose_values = Counter()
sample_items = []
total_items = 0
 
with open(FILE_PATH, 'rb') as f:
    for idx, item in enumerate(ijson.items(f, "item"), 1):
        total_items += 1
        
        # Get purpose value
        purpose = item.get('purpose')
        
        # Count it
        if purpose is not None:
            purpose_values[purpose] += 1
        else:
            purpose_values['NULL'] += 1
        
        # Store first 5 samples
        if len(sample_items) < 5:
            sample_items.append({
                'record': idx,
                'purpose': purpose,
                'title': item.get('title', 'N/A')[:50],
                'location': item.get('location', 'N/A')
            })
        
        # Progress
        if idx % 50000 == 0:
            print(f"  Scanned {idx:,} items...")
 
print(f"\n✓ Scanned {total_items:,} items total\n")
 
# Display results
print("="*80)
print("UNIQUE VALUES IN 'PURPOSE' FIELD")
print("="*80)
 
print(f"\nTotal unique values: {len(purpose_values)}\n")
 
# Sort by frequency
sorted_values = sorted(purpose_values.items(), key=lambda x: x[1], reverse=True)
 
for rank, (value, count) in enumerate(sorted_values, 1):
    percentage = (count / total_items * 100)
    bar = "█" * int(percentage / 2)  # Visual bar
    print(f"{rank}. {str(value):<30} Count: {count:>7,} ({percentage:>6.2f}%) {bar}")
 
print("\n" + "="*80)
print("SAMPLE RECORDS")
print("="*80)
 
for sample in sample_items:
    print(f"\nRecord #{sample['record']}:")
    print(f"  Purpose: {sample['purpose']}")
    print(f"  Title: {sample['title']}")
    print(f"  Location: {sample['location']}")
 
print("\n" + "="*80)
print("ANALYSIS")
print("="*80)
 
# Check if all same value
if len(purpose_values) == 1:
    only_value = list(purpose_values.keys())[0]
    print(f"\n⚠️  ALL {total_items:,} items have the SAME value:")
    print(f"   purpose = '{only_value}'")
else:
    print(f"\n✓ Found {len(purpose_values)} different values in 'purpose' field")
    print(f"\nMost common value:")
    top_value, top_count = sorted_values[0]
    print(f"  '{top_value}' appears {top_count:,} times ({top_count/total_items*100:.2f}%)")
 
print("\n" + "="*80)


CHECKING 'PURPOSE' FIELD VALUES
  Scanned 50,000 items...
  Scanned 100,000 items...

✓ Scanned 118,611 items total

UNIQUE VALUES IN 'PURPOSE' FIELD

Total unique values: 1

1. for-sale                       Count: 118,611 (100.00%) ██████████████████████████████████████████████████

SAMPLE RECORDS

Record #1:
  Purpose: for-sale
  Title: High ROI | Closed Kitchen | Fully Furnished | High
  Location: [{'id': 1, 'level': 0, 'externalID': '5001', 'name': 'UAE', 'name_l1': 'الإمارات', 'name_l2': '阿联酋', 'name_l3': 'ОАЭ', 'slug': '/uae'}, {'id': 2, 'level': 1, 'externalID': '5002', 'name': 'Dubai', 'name_l1': 'دبي', 'name_l2': '迪拜', 'name_l3': 'Дубай', 'slug': '/dubai'}, {'id': 59, 'level': 2, 'externalID': '5416', 'name': 'Jumeirah Village Circle (JVC)', 'name_l1': 'قرية جميرا الدائرية', 'name_l2': '朱美拉环形村(JVC)', 'name_l3': 'Джумейра Вилладж Серкл (ДЖВС)', 'slug': '/dubai/jumeirah-village-circle-jvc', 'type': 'neighbourhood'}, {'id': 1935, 'level': 3, 'externalID': '11385', 'name': 'JVC 

In [36]:
import ijson
import json
import pandas as pd
import sys
 
FILE_PATH = "/home/user/Downloads/bayut_properties_buy_residential_2026_08_19.json"
SEARCH_ID = 12279660  # ← CHANGE THIS TO YOUR ID
 
print("\n" + "="*80)
print(f"SEARCHING FOR ID: {SEARCH_ID}")
print("="*80)
 
found = None
found_at = None
 
try:
    with open(FILE_PATH, 'rb') as f:
        for rnum, item in enumerate(ijson.items(f, "item"), 1):
            if item.get('id') == SEARCH_ID:
                found = item
                found_at = rnum
                break
            
            if rnum % 50000 == 0:
                print(f"  Scanned {rnum:,} records... not found yet")
except Exception as e:
    print(f"❌ Error reading file: {e}")
    sys.exit(1)
 
if found:
    print(f"\n✅ FOUND at record #{found_at}\n")
    
    # Display key information
    print("=" * 80)
    print("KEY INFORMATION")
    print("=" * 80)
    
    info = {
        'Record Number': found_at,
        'ID': found.get('id'),
        'objectID': found.get('objectID'),
        'Title': found.get('title'),
        'Location': found.get('location'),
        'Property Type': found.get('property_type'),
        'Bedrooms': found.get('bedrooms'),
        'Bathrooms': found.get('bathrooms'),
        'Area': found.get('area'),
        'Price': f"{found.get('price')} {found.get('currency')}",
        'Listing URL': found.get('listing_url'),
        'Cover Photo': (found.get('coverPhoto')[:80] + "...") if found.get('coverPhoto') else 'None',
        'Total Images': len(found.get('images', [])),
    }
    
    for key, value in info.items():
        print(f"{key:<20}: {value}")
    
    # Display all fields
    print("\n" + "=" * 80)
    print("ALL FIELDS")
    print("=" * 80)
    
    for field in sorted(found.keys()):
        value = found[field]
        if value is None:
            print(f"  {field:<30}: NULL")
        elif isinstance(value, str) and len(value) > 80:
            print(f"  {field:<30}: {value[:80]}...")
        elif isinstance(value, list):
            print(f"  {field:<30}: [list with {len(value)} items]")
        elif isinstance(value, dict):
            print(f"  {field:<30}: {{dict with {len(value)} keys}}")
        else:
            print(f"  {field:<30}: {value}")
    
    # Export as JSON
    json_file = f"record_{SEARCH_ID}.json"
    try:
        with open(json_file, 'w') as f:
            json.dump(found, f, indent=2)
        print(f"\n✅ Exported to: {json_file}")
    except Exception as e:
        print(f"❌ Error exporting JSON: {e}")
    
    # Export as CSV (flattened)
    try:
        flat = found.copy()
        for key, value in flat.items():
            if isinstance(value, list):
                flat[key] = len(value)
            elif isinstance(value, dict):
                flat[key] = str(value)
        
        flat['record_number'] = found_at
        df = pd.DataFrame([flat])
        csv_file = f"record_{SEARCH_ID}.csv"
        df.to_csv(csv_file, index=False)
        print(f"✅ Exported to: {csv_file}")
    except Exception as e:
        print(f"❌ Error exporting CSV: {e}")
    
    print("\n" + "="*80)
    print("✅ RECORD FOUND AND EXPORTED")
    print("="*80 + "\n")
 
else:
    print(f"\n❌ NOT FOUND: No record with ID {SEARCH_ID}")
    print("\nPossible reasons:")
    print("  1. ID doesn't exist in dataset")
    print("  2. ID is in 'objectID' field, not 'id' field")
    print("  3. File path is incorrect")
    print("  4. JSON file is corrupted\n")
 






SEARCHING FOR ID: 12279660

✅ FOUND at record #42967

KEY INFORMATION
Record Number       : 42967
ID                  : 12279660
objectID            : 12279660
Title               : Stunning 1-Bed | Prime Location | Modern Finishes
Location            : [{'id': 1, 'level': 0, 'externalID': '5001', 'name': 'UAE', 'name_l1': 'الإمارات', 'name_l2': '阿联酋', 'name_l3': 'ОАЭ', 'slug': '/uae'}, {'id': 2, 'level': 1, 'externalID': '5002', 'name': 'Dubai', 'name_l1': 'دبي', 'name_l2': '迪拜', 'name_l3': 'Дубай', 'slug': '/dubai'}, {'id': 59, 'level': 2, 'externalID': '5416', 'name': 'Jumeirah Village Circle (JVC)', 'name_l1': 'قرية جميرا الدائرية', 'name_l2': '朱美拉环形村(JVC)', 'name_l3': 'Джумейра Вилладж Серкл (ДЖВС)', 'slug': '/dubai/jumeirah-village-circle-jvc', 'type': 'neighbourhood'}, {'id': 3327, 'level': 3, 'externalID': '11454', 'name': 'JVC District 14', 'name_l1': 'الضاحية 14', 'name_l2': 'JVC第14区', 'name_l3': 'JVC Дистрикт 14', 'slug': '/dubai/jumeirah-village-circle-jvc/jvc-district-14'

In [ ]:
# import ijson

# file_path = "/home/user/Downloads/bayut_properties_rent_residential_2026_08_19.json"

# fields = set()

# with open(file_path, "rb") as f:
#     for record in ijson.items(f, "item"):
#         fields.update(record.keys())

# print("Fields:")
# for field in sorted(fields):
#     print(field)

Fields:
active
agency
agentAdStoriesCount
amenities
approvedAt
area
availabilityStatus
baths
category
cityLevelScore
clips
completionDetails
completionStatus
contactMethodAvailability
contactName
coverPhoto
coverVideo
createdAt
description
descriptionTranslated
descriptionTranslated_l1
descriptionTranslated_l2
descriptionTranslated_l3
description_l1
description_l10
description_l11
description_l12
description_l13
description_l14
description_l2
description_l3
description_l4
description_l5
description_l6
description_l7
description_l8
description_l9
directFromOwner
externalID
extraFields
floorPlan
floorPlans
furnishingStatus
geography
hasExactGeography
hasMatchingFloorPlans
hidePrice
id
indyScore
indyScore_l1
indyScore_l2
indyScore_l3
isBusinessCenter
isHotelApartment
isShortTermRental
isVerified
listing_url
location
locationPurposeTier
objectID
occupancyStatus
ownerID
panoramaCount
panoramas
paymentPlanSummaries
paymentPlans
permitNumber
phoneNumber
photoCount
photos
plotArea
price
produc

In [4]:
import ijson
import re
import os
import csv
from collections import Counter, defaultdict
from urllib.parse import urlparse

# ============================================================
# CONFIG
# ============================================================

FILE_PATH = "/home/user/Downloads/bayut_properties_rent_residential_2026_08_19.json"

OUTPUT_DIR = "/home/user/Downloads/bayut_QA_report"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Description fields to exclude from value counts
DESCRIPTION_FIELDS = {
    "description",
    "descriptionTranslated",
    "descriptionTranslated_l1",
    "descriptionTranslated_l2",
    "descriptionTranslated_l3",
    "description_l1",
    "description_l2",
    "description_l3",
    "description_l4",
    "description_l5",
    "description_l6",
    "description_l7",
    "description_l8",
    "description_l9",
    "description_l10",
    "description_l11",
    "description_l12",
    "description_l13",
    "description_l14",
}

# URL fields
URL_FIELDS = {
    "listing_url",
    "coverPhoto",
}

# Image fields
IMAGE_FIELDS = {
    "coverPhoto",
    "photos",
}

# ============================================================
# STORAGE
# ============================================================

total_records = 0

all_fields = set()

field_count = Counter()
null_count = Counter()
empty_count = Counter()

value_counts = defaultdict(Counter)
type_counts = defaultdict(Counter)

duplicate_ids = Counter()
duplicate_listing_urls = Counter()

whitespace_issues = []
pipe_issues = []
html_issues = []

invalid_url_format = []
invalid_image_format = []

unexpected_fields = []

MAX_EXAMPLES = 5000


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def is_valid_url(value):

    if not isinstance(value, str):
        return False

    try:
        parsed = urlparse(value)

        return (
            parsed.scheme in ("http", "https")
            and bool(parsed.netloc)
        )

    except Exception:
        return False


def is_empty(value):

    if value is None:
        return True

    if isinstance(value, str):
        return value.strip() == ""

    if isinstance(value, list):
        return len(value) == 0

    if isinstance(value, dict):
        return len(value) == 0

    return False


def check_string(row_no, field, value):

    # Whitespace
    if value != value.strip():

        if len(whitespace_issues) < MAX_EXAMPLES:
            whitespace_issues.append({
                "row": row_no,
                "field": field,
                "issue": "Leading/trailing whitespace",
                "value": value
            })

    # Multiple spaces
    if re.search(r"[ \t]{2,}", value):

        if len(whitespace_issues) < MAX_EXAMPLES:
            whitespace_issues.append({
                "row": row_no,
                "field": field,
                "issue": "Multiple whitespace",
                "value": value
            })

    # Newline
    if "\n" in value or "\r" in value:

        if len(whitespace_issues) < MAX_EXAMPLES:
            whitespace_issues.append({
                "row": row_no,
                "field": field,
                "issue": "Newline character",
                "value": value
            })

    # Tab
    if "\t" in value:

        if len(whitespace_issues) < MAX_EXAMPLES:
            whitespace_issues.append({
                "row": row_no,
                "field": field,
                "issue": "Tab character",
                "value": value
            })

    # Pipe
    if "|" in value:

        if len(pipe_issues) < MAX_EXAMPLES:
            pipe_issues.append({
                "row": row_no,
                "field": field,
                "value": value
            })

    # HTML tags
    if re.search(r"<[^>]+>", value):

        if len(html_issues) < MAX_EXAMPLES:
            html_issues.append({
                "row": row_no,
                "field": field,
                "value": value
            })


def inspect_nested(value, row_no, parent_field):

    """
    Recursively inspect nested dictionaries/lists.
    """

    if isinstance(value, dict):

        for key, val in value.items():

            field_name = f"{parent_field}.{key}"

            if isinstance(val, str):
                check_string(row_no, field_name, val)

            inspect_nested(
                val,
                row_no,
                field_name
            )

    elif isinstance(value, list):

        for index, item in enumerate(value):

            field_name = f"{parent_field}[{index}]"

            if isinstance(item, str):
                check_string(row_no, field_name, item)

            inspect_nested(
                item,
                row_no,
                field_name
            )


def extract_image_urls(value):

    """
    Recursively find possible image URLs.
    """

    urls = []

    if isinstance(value, str):

        if is_valid_url(value):
            urls.append(value)

    elif isinstance(value, list):

        for item in value:
            urls.extend(
                extract_image_urls(item)
            )

    elif isinstance(value, dict):

        for key, val in value.items():

            key_lower = key.lower()

            if (
                "url" in key_lower
                or "image" in key_lower
                or "photo" in key_lower
            ):

                if isinstance(val, str):
                    urls.append(val)

            urls.extend(
                extract_image_urls(val)
            )

    return urls


# ============================================================
# PROCESS JSON
# ============================================================

print("Starting QA...")
print("-" * 70)

with open(FILE_PATH, "rb") as f:

    records = ijson.items(f, "item")

    for row_no, record in enumerate(records, start=1):

        total_records += 1

        if not isinstance(record, dict):
            continue

        # ----------------------------------------------------
        # FIELD NAMES
        # ----------------------------------------------------

        current_fields = set(record.keys())

        all_fields.update(current_fields)

        for field in current_fields:
            field_count[field] += 1

        # ----------------------------------------------------
        # DUPLICATE ID
        # ----------------------------------------------------

        record_id = record.get("id")

        if record_id not in (None, ""):
            duplicate_ids[record_id] += 1

        # ----------------------------------------------------
        # DUPLICATE LISTING URL
        # ----------------------------------------------------

        listing_url = record.get("listing_url")

        if listing_url not in (None, ""):
            duplicate_listing_urls[listing_url] += 1

        # ----------------------------------------------------
        # FIELD QA
        # ----------------------------------------------------

        for field, value in record.items():

            # Data type
            type_counts[field][
                type(value).__name__
            ] += 1

            # Missing
            if value is None:

                null_count[field] += 1

            # Empty
            elif is_empty(value):

                empty_count[field] += 1

            # ------------------------------------------------
            # VALUE COUNT
            # ------------------------------------------------

            if field not in DESCRIPTION_FIELDS:

                if isinstance(
                    value,
                    (str, int, float, bool)
                ):

                    try:
                        value_counts[field][
                            value
                        ] += 1

                    except TypeError:
                        pass

            # ------------------------------------------------
            # STRING CHECKS
            # ------------------------------------------------

            if isinstance(value, str):

                check_string(
                    row_no,
                    field,
                    value
                )

            # ------------------------------------------------
            # URL CHECK
            # ------------------------------------------------

            if field in URL_FIELDS:

                if value not in (None, ""):

                    if not is_valid_url(value):

                        if len(invalid_url_format) < MAX_EXAMPLES:

                            invalid_url_format.append({
                                "row": row_no,
                                "field": field,
                                "url": value,
                                "issue": "Invalid URL format"
                            })

            # ------------------------------------------------
            # IMAGE CHECK
            # ------------------------------------------------

            if field in IMAGE_FIELDS:

                image_urls = extract_image_urls(value)

                for image_url in image_urls:

                    if not is_valid_url(image_url):

                        if len(invalid_image_format) < MAX_EXAMPLES:

                            invalid_image_format.append({
                                "row": row_no,
                                "field": field,
                                "url": image_url,
                                "issue": "Invalid image URL"
                            })

            # ------------------------------------------------
            # NESTED DATA
            # ------------------------------------------------

            if isinstance(value, (dict, list)):

                inspect_nested(
                    value,
                    row_no,
                    field
                )

        # ----------------------------------------------------
        # PROGRESS
        # ----------------------------------------------------

        if row_no % 10000 == 0:

            print(
                f"Processed {row_no:,} records..."
            )


print("-" * 70)
print("QA processing completed.")
print("Total records:", total_records)
print("Total fields:", len(all_fields))


# ============================================================
# DUPLICATES
# ============================================================

duplicate_id_results = []

for key, count in duplicate_ids.items():

    if count > 1:

        duplicate_id_results.append({
            "id": key,
            "count": count
        })


duplicate_url_results = []

for key, count in duplicate_listing_urls.items():

    if count > 1:

        duplicate_url_results.append({
            "listing_url": key,
            "count": count
        })


# ============================================================
# FIELD SUMMARY
# ============================================================

field_summary = []

for field in sorted(all_fields):

    field_summary.append({
        "field": field,
        "present_count": field_count[field],
        "missing_count": total_records - field_count[field],
        "null_count": null_count[field],
        "empty_count": empty_count[field],
        "data_types": str(
            dict(type_counts[field])
        )
    })


# ============================================================
# VALUE COUNTS
# ============================================================

value_count_results = []

for field, counter in value_counts.items():

    for value, count in counter.most_common():

        value_count_results.append({
            "field": field,
            "value": value,
            "count": count
        })


# ============================================================
# CSV WRITER
# ============================================================

def save_csv(filename, rows, columns):

    path = os.path.join(
        OUTPUT_DIR,
        filename
    )

    with open(
        path,
        "w",
        newline="",
        encoding="utf-8"
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=columns
        )

        writer.writeheader()

        writer.writerows(rows)

    print("Created:", path)


# ============================================================
# SAVE RESULTS
# ============================================================

save_csv(
    "field_summary.csv",
    field_summary,
    [
        "field",
        "present_count",
        "missing_count",
        "null_count",
        "empty_count",
        "data_types"
    ]
)


save_csv(
    "value_counts.csv",
    value_count_results,
    [
        "field",
        "value",
        "count"
    ]
)


save_csv(
    "duplicate_ids.csv",
    duplicate_id_results,
    [
        "id",
        "count"
    ]
)


save_csv(
    "duplicate_listing_urls.csv",
    duplicate_url_results,
    [
        "listing_url",
        "count"
    ]
)


save_csv(
    "whitespace_issues.csv",
    whitespace_issues,
    [
        "row",
        "field",
        "issue",
        "value"
    ]
)


save_csv(
    "pipe_issues.csv",
    pipe_issues,
    [
        "row",
        "field",
        "value"
    ]
)


save_csv(
    "html_tag_issues.csv",
    html_issues,
    [
        "row",
        "field",
        "value"
    ]
)


save_csv(
    "invalid_url_format.csv",
    invalid_url_format,
    [
        "row",
        "field",
        "url",
        "issue"
    ]
)


save_csv(
    "invalid_image_urls.csv",
    invalid_image_format,
    [
        "row",
        "field",
        "url",
        "issue"
    ]
)


# ============================================================
# TEXT REPORT
# ============================================================

report_path = os.path.join(
    OUTPUT_DIR,
    "QA_report.txt"
)

with open(
    report_path,
    "w",
    encoding="utf-8"
) as f:

    f.write("BAYUT JSON QA REPORT\n")
    f.write("=" * 70 + "\n\n")

    f.write(
        f"Total records: {total_records:,}\n"
    )

    f.write(
        f"Total fields: {len(all_fields)}\n\n"
    )

    f.write("ISSUE SUMMARY\n")
    f.write("-" * 70 + "\n")

    f.write(
        f"Duplicate IDs: "
        f"{len(duplicate_id_results)}\n"
    )

    f.write(
        f"Duplicate listing URLs: "
        f"{len(duplicate_url_results)}\n"
    )

    f.write(
        f"Whitespace issues: "
        f"{len(whitespace_issues)}\n"
    )

    f.write(
        f"Pipe character issues: "
        f"{len(pipe_issues)}\n"
    )

    f.write(
        f"HTML tag issues: "
        f"{len(html_issues)}\n"
    )

    f.write(
        f"Invalid URL format: "
        f"{len(invalid_url_format)}\n"
    )

    f.write(
        f"Invalid image URLs: "
        f"{len(invalid_image_format)}\n"
    )


print()
print("=" * 70)
print("DONE")
print("=" * 70)
print("Output folder:")
print(OUTPUT_DIR)

Starting QA...
----------------------------------------------------------------------
Processed 10,000 records...
Processed 20,000 records...
Processed 30,000 records...
Processed 40,000 records...
Processed 50,000 records...
Processed 60,000 records...
Processed 70,000 records...
Processed 80,000 records...
Processed 90,000 records...
Processed 100,000 records...
Processed 110,000 records...
Processed 120,000 records...
Processed 130,000 records...
----------------------------------------------------------------------
QA processing completed.
Total records: 138781
Total fields: 111
Created: /home/user/Downloads/bayut_QA_report/field_summary.csv
Created: /home/user/Downloads/bayut_QA_report/value_counts.csv
Created: /home/user/Downloads/bayut_QA_report/duplicate_ids.csv
Created: /home/user/Downloads/bayut_QA_report/duplicate_listing_urls.csv
Created: /home/user/Downloads/bayut_QA_report/whitespace_issues.csv
Created: /home/user/Downloads/bayut_QA_report/pipe_issues.csv
Created: /home/u

In [6]:
import ijson
import pandas as pd

file_path = "/home/user/Downloads/bayut_properties_buy_commercial_2026_08_19.json"

results = []

with open(file_path, "rb") as f:

    records = ijson.items(f, "item")

    for row_no, record in enumerate(records, start=1):

        cover_photo = record.get("coverPhoto")

        # Check missing / empty coverPhoto
        if (
            cover_photo is None
            or cover_photo == ""
            or (
                isinstance(cover_photo, str)
                and cover_photo.strip() == ""
            )
        ):

            results.append({
                "record_no": row_no,
                "listing_url": record.get("listing_url"),
                "coverPhoto": cover_photo
            })

print("Total records with empty/missing coverPhoto:", len(results))

df_missing_coverphoto = pd.DataFrame(results)

print(df_missing_coverphoto)

Total records with empty/missing coverPhoto: 0
Empty DataFrame
Columns: []
Index: []


In [7]:
import ijson
import csv
import os
from collections import Counter

file_path = "/home/user/Downloads/bayut_properties_buy_commercial_2026_08_19.json"

output_file = "/home/user/Downloads/bayut_field_value_counts.csv"

# Fields where full descriptions are not useful for value counting
EXCLUDE_FIELDS = {
    "description",
    "descriptionTranslated",
    "descriptionTranslated_l1",
    "descriptionTranslated_l2",
    "descriptionTranslated_l3",
    "description_l1",
    "description_l2",
    "description_l3",
    "description_l4",
    "description_l5",
    "description_l6",
    "description_l7",
    "description_l8",
    "description_l9",
    "description_l10",
    "description_l11",
    "description_l12",
    "description_l13",
    "description_l14",
}

# Store value counts
value_counts = {}

total_records = 0

print("Reading JSON...")

with open(file_path, "rb") as f:

    records = ijson.items(f, "item")

    for record_no, record in enumerate(records, start=1):

        total_records += 1

        for field, value in record.items():

            if field in EXCLUDE_FIELDS:
                continue

            # Create counter for field
            if field not in value_counts:
                value_counts[field] = Counter()

            # Convert complex values to string
            if isinstance(value, (dict, list)):

                # Keep only non-empty complex values
                if value:
                    value_string = str(value)
                    value_counts[field][value_string] += 1

            else:

                # Keep NULL also
                if value is None:
                    value_string = "NULL"
                else:
                    value_string = str(value)

                value_counts[field][value_string] += 1

        if record_no % 5000 == 0:
            print(f"Processed {record_no:,} records")


print("\nTotal records:", total_records)
print("Creating value count file...")


# ---------------------------------------------------------
# SAVE VALUE COUNTS
# ---------------------------------------------------------

with open(
    output_file,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.writer(f)

    writer.writerow([
        "field",
        "value",
        "count"
    ])

    for field in sorted(value_counts):

        counter = value_counts[field]

        for value, count in counter.most_common():

            writer.writerow([
                field,
                value,
                count
            ])


print("\nDone!")
print("Output:", output_file)

Reading JSON...

Total records: 3619
Creating value count file...

Done!
Output: /home/user/Downloads/bayut_field_value_counts.csv


In [9]:
import ijson

file_path = "/home/user/Downloads/bayut_properties_buy_commercial_2026_08_19.json.json"

with open(file_path, "rb") as f:

    for record_no, record in enumerate(
        ijson.items(f, "item"),
        start=1
    ):

        floor_plan = record.get("floorPlan")

        if floor_plan is not None:

            print("Record No:", record_no)
            print("Listing URL:", record.get("listing_url"))
            print("Floor Plan:")
            print(floor_plan)
            print("-" * 80)

FileNotFoundError: [Errno 2] No such file or directory: '/home/user/Downloads/bayut_properties_buy_commercial_2026_08_19.json.json'

In [15]:
import ijson

file_path = "/home/user/Downloads/bayut_properties_buy_commercial_2026_08_19.json"

count = 0

with open(file_path, "rb") as f:

    for record_no, record in enumerate(
        ijson.items(f, "item"),
        start=1
    ):

        if record.get("furnishingStatus") is None:

            print(
                f"Record No: {record_no} | "
                f"Listing URL: {record.get('listing_url')} | "
                f"furnishingStatus: {record.get('furnishingStatus')}"
            )

            count += 1

            if count == 5:
                break

Record No: 3 | Listing URL: https://www.bayut.com/property/details-16160961.html | furnishingStatus: None
Record No: 50 | Listing URL: https://www.bayut.com/property/details-16174740.html | furnishingStatus: None
Record No: 52 | Listing URL: https://www.bayut.com/property/details-16130866.html | furnishingStatus: None
Record No: 71 | Listing URL: https://www.bayut.com/property/details-16140299.html | furnishingStatus: None
Record No: 72 | Listing URL: https://www.bayut.com/property/details-15465280.html | furnishingStatus: None


In [14]:
import ijson
import pandas as pd

file_path = "/home/user/Downloads/bayut_properties_rent_commercial_2026_08_19.json"

results = []

with open(file_path, "rb") as f:

    for record_no, record in enumerate(
        ijson.items(f, "item"),
        start=1
    ):

        furnishing_status = record.get("furnishingStatus")

        if furnishing_status is not None:

            results.append({
                "record_no": record_no,
                "listing_url": record.get("listing_url"),
                "furnishingStatus": furnishing_status
            })

            if len(results) == 5:
                break

df = pd.DataFrame(results)

print(df.to_string(index=False))

 record_no                                          listing_url furnishingStatus
         1 https://www.bayut.com/property/details-14679295.html        furnished
         2 https://www.bayut.com/property/details-12092852.html        furnished
         3 https://www.bayut.com/property/details-12439222.html        furnished
         4  https://www.bayut.com/property/details-7493237.html        furnished
         5 https://www.bayut.com/property/details-11931261.html        furnished


In [6]:
def extract_location_names(location):
    if isinstance(location, list):
        return [x.get("name") for x in location if isinstance(x, dict)]
    return []

df["location_names"] = df["location"].apply(extract_location_names)

df[["location", "location_names"]].head()

,location,location_names
0,"[{'id': 1, 'level': 0, 'externalID': '5001', 'name': 'UAE', 'name_l1': 'الإمارات', 'name_l2': '阿联酋', 'name_l3': 'ОАЭ', 'slug': '/uae'}, {'id': 2, 'level': 1, 'externalID': '5002', 'name': 'Dubai', 'name_l1': 'دبي', 'name_l2': '迪拜', 'name_l3': 'Дубай', 'slug': '/dubai'}, {'id': 345, 'level': 2, 'externalID': '6796', 'name': 'Bur Dubai', 'name_l1': 'بر دبي', 'name_l2': '迪拜湾', 'name_l3': 'Бур Дубай', 'slug': '/dubai/bur-dubai', 'type': 'neighbourhood'}, {'id': 2711, 'level': 3, 'externalID': '5638', 'name': 'Al Hamriya', 'name_l1': 'الحمریة', 'name_l2': '哈姆利亚街区', 'name_l3': 'Аль Хамрия', 'slug': '/dubai/bur-dubai/al-hamriya', 'type': 'neighbourhood'}, {'id': 2712, 'level': 4, 'externalID': '5697', 'name': 'Fahidi Heights', 'name_l1': 'الفهيدي هايتس', 'name_l2': '穆萨拉大厦', 'name_l3': 'Здание Аль Мусалла', 'slug': '/dubai/bur-dubai/al-hamriya/fahidi-heights', 'type': 'condo-building'}]","[UAE, Dubai, Bur Dubai, Al Hamriya, Fahidi Heights]"
1,"[{'id': 1, 'level': 0, 'externalID': '5001', 'name': 'UAE', 'name_l1': 'الإمارات', 'name_l2': '阿联酋', 'name_l3': 'ОАЭ', 'slug': '/uae'}, {'id': 2, 'level': 1, 'externalID': '5002', 'name': 'Dubai', 'name_l1': 'دبي', 'name_l2': '迪拜', 'name_l3': 'Дубай', 'slug': '/dubai'}, {'id': 345, 'level': 2, 'externalID': '6796', 'name': 'Bur Dubai', 'name_l1': 'بر دبي', 'name_l2': '迪拜湾', 'name_l3': 'Бур Дубай', 'slug': '/dubai/bur-dubai', 'type': 'neighbourhood'}, {'id': 2711, 'level': 3, 'externalID': '5638', 'name': 'Al Hamriya', 'name_l1': 'الحمریة', 'name_l2': '哈姆利亚街区', 'name_l3': 'Аль Хамрия', 'slug': '/dubai/bur-dubai/al-hamriya', 'type': 'neighbourhood'}, {'id': 2712, 'level': 4, 'externalID': '5697', 'name': 'Fahidi Heights', 'name_l1': 'الفهيدي هايتس', 'name_l2': '穆萨拉大厦', 'name_l3': 'Здание Аль Мусалла', 'slug': '/dubai/bur-dubai/al-hamriya/fahidi-heights', 'type': 'condo-building'}]","[UAE, Dubai, Bur Dubai, Al Hamriya, Fahidi Heights]"
2,"[{'id': 1, 'level': 0, 'externalID': '5001', 'name': 'UAE', 'name_l1': 'الإمارات', 'name_l2': '阿联酋', 'name_l3': 'ОАЭ', 'slug': '/uae'}, {'id': 2, 'level': 1, 'externalID': '5002', 'name': 'Dubai', 'name_l1': 'دبي', 'name_l2': '迪拜', 'name_l3': 'Дубай', 'slug': '/dubai'}, {'id': 345, 'level': 2, 'externalID': '6796', 'name': 'Bur Dubai', 'name_l1': 'بر دبي', 'name_l2': '迪拜湾', 'name_l3': 'Бур Дубай', 'slug': '/dubai/bur-dubai', 'type': 'neighbourhood'}, {'id': 2711, 'level': 3, 'externalID': '5638', 'name': 'Al Hamriya', 'name_l1': 'الحمریة', 'name_l2': '哈姆利亚街区', 'name_l3': 'Аль Хамрия', 'slug': '/dubai/bur-dubai/al-hamriya', 'type': 'neighbourhood'}, {'id': 2712, 'level': 4, 'externalID': '5697', 'name': 'Fahidi Heights', 'name_l1': 'الفهيدي هايتس', 'name_l2': '穆萨拉大厦', 'name_l3': 'Здание Аль Мусалла', 'slug': '/dubai/bur-dubai/al-hamriya/fahidi-heights', 'type': 'condo-building'}]","[UAE, Dubai, Bur Dubai, Al Hamriya, Fahidi Heights]"
3,"[{'id': 1, 'level': 0, 'externalID': '5001', 'name': 'UAE', 'name_l1': 'الإمارات', 'name_l2': '阿联酋', 'name_l3': 'ОАЭ', 'slug': '/uae'}, {'id': 2, 'level': 1, 'externalID': '5002', 'name': 'Dubai', 'name_l1': 'دبي', 'name_l2': '迪拜', 'name_l3': 'Дубай', 'slug': '/dubai'}, {'id': 345, 'level': 2, 'externalID': '6796', 'name': 'Bur Dubai', 'name_l1': 'بر دبي', 'name_l2': '迪拜湾', 'name_l3': 'Бур Дубай', 'slug': '/dubai/bur-dubai', 'type': 'neighbourhood'}, {'id': 2711, 'level': 3, 'externalID': '5638', 'name': 'Al Hamriya', 'name_l1': 'الحمریة', 'name_l2': '哈姆利亚街区', 'name_l3': 'Аль Хамрия', 'slug': '/dubai/bur-dubai/al-hamriya', 'type': 'neighbourhood'}, {'id': 2712, 'level': 4, 'externalID': '5697', 'name': 'Fahidi Heights', 'name_l1': 'الفهيدي هايتس', 'name_l2': '穆萨拉大厦', 'name_l3': 'Здание Аль Мусалла', 'slug': '/dubai/bur-dubai/al-hamriya/fahidi-heights', 'type': 'condo-building'}]","[UAE, Dubai, Bur Dubai, Al Hamriya, Fahidi Heights]"
4,"[{'id': 1, 'level': 0, 'externalID': '5001', 'name': 'UAE', 'name_l1': 'الإمارات', 'name_l2': '阿联酋', 'name_l3': 'ОАЭ', 'slug': '/uae'}, {'id': 2, 'level': 1, 'externalID': 

In [7]:
# Extract location names
def extract_location_names(location):
    if isinstance(location, list):
        return [x.get("name") for x in location if isinstance(x, dict)]
    return []

df["location_names"] = df["location"].apply(extract_location_names)

# Find records where Dubai is missing
invalid_location = df[
    ~df["location_names"].apply(lambda x: "Dubai" in x)
]

# Add record number
invalid_location = invalid_location.copy()
invalid_location["record_no"] = invalid_location.index + 1

# Display required columns
result = invalid_location[
    ["record_no", "listing_url", "location_names"]
]

print(result)

Empty DataFrame
Columns: [record_no, listing_url, location_names]
Index: []


In [12]:
import ijson
from collections import Counter

file_path = "/home/user/Downloads/bayut_properties_buy_commercial_2026_08_19.json"

location_counts = Counter()
purpose_counts = Counter()

invalid_location_records = []
invalid_purpose_records = []

with open(file_path, "rb") as f:

    for record_no, record in enumerate(
        ijson.items(f, "item"),
        start=1
    ):

        # ----------------------------------------
        # LOCATION VALUES
        # ----------------------------------------

        location = record.get("location", [])

        location_names = []

        if isinstance(location, list):

            for item in location:

                if isinstance(item, dict):

                    name = item.get("name")

                    if name:
                        location_names.append(name)

        for name in location_names:
            location_counts[name] += 1

        # Check only Dubai location
        if not any(
            name.strip().lower() == "dubai"
            for name in location_names
        ):

            invalid_location_records.append({
                "record_no": record_no,
                "listing_url": record.get("listing_url"),
                "location": location_names
            })

        # ----------------------------------------
        # PURPOSE
        # ----------------------------------------

        purpose = record.get("purpose")

        if purpose is not None:
            purpose_counts[purpose] += 1

        # Only sale should be present
        if str(purpose).strip().lower() != "for-sale":

            invalid_purpose_records.append({
                "record_no": record_no,
                "listing_url": record.get("listing_url"),
                "purpose": purpose
            })


# ============================================================
# RESULTS
# ============================================================

print("\nLOCATION VALUES")
print("=" * 60)

for location, count in location_counts.most_common():
    print(f"{location}: {count}")


print("\nPURPOSE VALUES")
print("=" * 60)

for purpose, count in purpose_counts.most_common():
    print(f"{purpose}: {count}")


print("\nLOCATION QA")
print("=" * 60)

print(
    "Records not having Dubai location:",
    len(invalid_location_records)
)


print("\nPURPOSE QA")
print("=" * 60)

print(
    "Records where purpose is not 'for-sale':",
    len(invalid_purpose_records)
)


# Show invalid purpose records
for record in invalid_purpose_records[:20]:

    print(
        record["record_no"],
        "|",
        record["purpose"],
        "|",
        record["listing_url"]
    )


LOCATION VALUES
UAE: 3619
Dubai: 3619
Business Bay: 813
Jumeirah Lake Towers (JLT): 380
Jebel Ali: 370
Jumeirah Village Circle (JVC): 325
Majan: 201
Downtown Jebel Ali: 174
Jebel Ali Freezone: 168
Barsha Heights (Tecom): 127
Dubai Investments Park (DIP): 120
Arjan: 113
JVC District 13: 106
Dubai Silicon Oasis (DSO): 100
International City: 98
Meydan: 96
Samana Barari Avenue: 91
Motor City: 83
Samana Business Hub: 76
Dubai Land Residence Complex (DLRC): 76
DIFC: 72
Burj Capital: 70
Meydan One: 66
Azizi Riviera: 66
Binghatti Amberhall: 62
HQ by Rove: 58
Al Sufouh: 56
Al Sufouh 1: 56
Meydan Horizon: 54
Shahrukhz by Danube: 54
Al Jaddaf: 54
Raw District II by Imtiaz: 49
Al Furjan: 46
JLT Cluster N: 45
Tamani Arts Offices: 45
I-Rise Tower: 45
Jebel Ali Freezone South: 44
JVC District 11: 41
Jebel Ali Freezone North: 40
JLT Cluster X (Jumeirah Bay Towers): 39
JVC District 16: 39
JLT Cluster W: 38
Sheikh Zayed Road: 37
Dubai Sports City: 36
JVC District 10: 36
Emirates Financial Towers: 35
A

In [16]:
import ijson
from collections import Counter

file_path = "/home/user/Downloads/bayut_properties_buy_commercial_2026_08_19.json"

purpose_counts = Counter()

with open(file_path, "rb") as f:

    for record in ijson.items(f, "item"):

        purpose = record.get("purpose")

        if purpose is None:
            purpose = "NULL"

        purpose_counts[str(purpose)] += 1


print("Purpose values:")
print("-" * 40)

for value, count in purpose_counts.most_common():
    print(f"{value}: {count}")

Purpose values:
----------------------------------------
for-sale: 3619


In [17]:
import re
import pandas as pd

def get_example(mask):
    rows = df[mask]
    
    if len(rows) > 0:
        row = rows.iloc[0]
        return (
            f"Record No: {rows.index[0] + 1} | "
            f"Listing URL: {row['listing_url']}"
        )
    
    return "No issue found"


# 1. HTML <strong> tag
strong_mask = (
    df["description"]
    .fillna("")
    .astype(str)
    .str.contains(r"<strong>", case=False, regex=True)
)

# 2. Newline characters
newline_mask = (
    df["description"]
    .fillna("")
    .astype(str)
    .str.contains(r"[\r\n]", regex=True)
)

# 3. Unescaped |
# Change this condition if your expected escaping format is different
pipe_mask = (
    df["description"]
    .fillna("")
    .astype(str)
    .str.contains(r"(?<!\\)\|", regex=True)
)

# 4. Extra whitespace
whitespace_mask = (
    df["description"]
    .fillna("")
    .astype(str)
    .str.contains(r"(^\s+|\s+$|\s{2,})", regex=True)
)


print("- HTML tag <strong> found in the description field, but not present on the website.")
print("  Example:", get_example(strong_mask))

print("\n- Newline characters found.")
print("  Example:", get_example(newline_mask))

print("\n- Unescaped | found.")
print("  Example:", get_example(pipe_mask))

print("\n- Extra white space found.")
print("  Example:", get_example(whitespace_mask))

KeyError: 'description'

In [18]:
import re
import pandas as pd

def get_example(mask):
    rows = df[mask]

    if len(rows) > 0:
        idx = rows.index[0]

        return (
            f"Record No: {idx + 1} | "
            f"Listing URL: {df.loc[idx, 'listing_url']}"
        )

    return "No issue found"


description = df["description"].fillna("").astype(str)


# HTML <strong> tag
strong_mask = description.str.contains(
    r"<strong>",
    case=False,
    regex=True
)


# Newline characters
newline_mask = description.str.contains(
    r"[\r\n]",
    regex=True
)


# Unescaped |
pipe_mask = description.str.contains(
    r"(?<!\\)\|",
    regex=True
)


# Extra whitespace
whitespace_mask = description.str.contains(
    r"(^\s+|\s+$|\s{2,})",
    regex=True
)


print(
    "- HTML tag <strong> found in the description field, "
    "but not present on the website."
)
print("  Example:", get_example(strong_mask))


print("\n- Newline characters found.")
print("  Example:", get_example(newline_mask))


print("\n- Unescaped | found.")
print("  Example:", get_example(pipe_mask))


print("\n- Extra white space found.")
print("  Example:", get_example(whitespace_mask))

KeyError: 'description'